# 3 · Evaluation — per-detector vs. fused (AUROC, AUPR, confusion matrices)

Headline metrics are **AUROC** and **AUPR** (threshold-free, the numbers the source papers report). Confusion matrices use each individual detector's best-F1 threshold and **0.5** for the calibrated fused model. Goal: **fused ≥ best individual**, in range of the published baselines.

In [ ]:
import os, sys
os.environ.setdefault('HF_HOME', r'D:/LLAMA CACHE/huggingface')  # reuse local LLaMA cache
sys.path.insert(0, os.path.abspath(os.path.join('..', 'hallking')))
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import metrics as M
DATASET='triviaqa'
test = pd.read_parquet(os.path.join('..','data',f'{DATASET}_eval_oof.parquet'))  # honest OOF, from nb2
y = test['hallucination'].to_numpy()
# detector score columns (higher = more hallucinated)
detectors = {
  'SEP':        test['sep_entropy'].to_numpy(),
  'HalluShift': test['hallushift'].to_numpy(),
  'TSV':        test['tsv_margin'].to_numpy(),
  'FUSED':      test['fused'].to_numpy(),
}

In [ ]:
res = {}
for name, s in detectors.items():
    thr = M.best_threshold(y, s, metric='f1')
    m = M.detector_metrics(y, s, threshold=thr)
    M.attach_curves(m, y, s)
    res[name] = m
summary = M.summary_table(res)
summary

In [ ]:
# paper-reported AUROC for context (LLaMA-3.1-8B, TriviaQA; from each paper's own generation)
paper = pd.DataFrame({'paper_AUROC_TriviaQA':{
    'SEP':'~0.78 (Llama-2 in-dist; no 3.1-8B number)',
    'HalluShift':0.99,   # their own run; our held-out reproduction is lower
    'TSV':0.84,          # tqa-trained, transfers to triviaqa ~0.80
    'FUSED':'(this work)'}})
paper

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(11,4))
M.plot_roc(ax[0], res); M.plot_pr(ax[1], res)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1,4, figsize=(15,3.4))
for ax,(name,m) in zip(axes, res.items()):
    M.plot_confusion(ax, m['confusion_matrix'], title=f"{name}\nF1={m['F1']:.2f} thr={m['threshold']:.2f}")
plt.tight_layout(); plt.show()

In [ ]:
# inter-detector correlation (low correlation => complementary => fusion helps)
test[['sep_entropy','hallushift','tsv_margin']].corr().round(2)